# 08 — Aspect-Guided Attention (Phase 1 + Phase 2)

Demos and ablations for the auxiliary aspect-conditioned attention heads: per-aspect score regression, evidence-span extraction, and faithfulness diagnostics. See `docs/ASPECT_ATTENTION_GUIDE.md` for the full scientific write-up.

**Requires:** `aspect.enabled: true` in the active config, and a checkpoint trained with `04_training.ipynb` (which saves `aspect_heads.pt` alongside the LoRA adapter when enabled).

In [ ]:
# Bootstrap path (required before import project)
exec(open("/content/ABPSEES/colab_bootstrap.py").read())
ROOT = setup_colab_path()

%load_ext autoreload
%autoreload 2

from project.config import load_config, is_colab
from project.utils import setup_logging, setup_colab_environment

setup_logging()
config = load_config()
setup_colab_environment(
    hf_token_env=config.colab.hf_token_env,
    use_colab_secrets=config.colab.use_colab_secrets,
    mount_google_drive=config.colab.mount_google_drive,
)

print('Project root:', ROOT)
print('Colab:', is_colab())
print('Data dir:', config.paths.data_dir)


In [ ]:
print('Aspect attention enabled:', config.aspect.enabled)
print('Hidden layer:', config.aspect.hidden_layer)
print('Lambdas: ce={} score={} span={} faith={}'.format(
    config.aspect.lambda_ce, config.aspect.lambda_score,
    config.aspect.lambda_span, config.aspect.lambda_faith,
))


## Load the hybrid model

Loads the base Qwen model + LoRA adapter + the auxiliary `AspectHeads` (score/span/has-evidence) from `outputs/checkpoints/best/`.

In [ ]:
from project.hybrid_model import load_hybrid_model_for_inference

hybrid_model, tokenizer = load_hybrid_model_for_inference(config)
print(hybrid_model.aspect_heads)


## Single-example demo: score head vs generated JSON

Compares the primary generative JSON prediction with the auxiliary score/span heads on one Persian example.

In [ ]:
from project.inference import predict, pretty_print_prediction

text = (
    'من دانشجوی مهندسی کامپیوتر هستم و می خواهم مدل‌های هوش مصنوعی آموزش بدهم. '
    'هر روز لپ‌تاپم را با خودم به دانشگاه می‌برم.'
)
result = predict(text, config, model=hybrid_model, tokenizer=tokenizer)
print(pretty_print_prediction(result))
print()
print('Aspect head diagnostics:')
for aspect, diag in result.get('_aspect_attention', {}).items():
    print(f'  {aspect}: score_head={diag["score_head"]:.3f} '
          f'has_evidence_prob={diag["has_evidence_prob"]:.3f} '
          f'span_evidence={diag["span_evidence"]!r}')


## Attention heatmap for one aspect

Visualizes the aspect-conditioned attention distribution `alpha_a` over the input tokens, for direct inspection of *where* the model is looking when scoring each aspect.

In [ ]:
from project.hybrid_model import run_aspect_heads_on_text
from project.visualization import plot_aspect_attention_heatmap

diag = run_aspect_heads_on_text(hybrid_model, tokenizer, text, config)
fig_path = plot_aspect_attention_heatmap(
    diag['tokens'], diag['attn_weights'],
    config.paths.reports_dir / 'figures' / 'attention' / 'demo_sample.png',
)
print('Saved heatmap to', fig_path)


## Full test-set aspect-head evaluation

Runs the same head-based evaluation used by `run_evaluation` (score MAE/RMSE, span Token F1/EM, faithfulness rates) standalone, for ablation experiments.

In [ ]:
from project.evaluation import evaluate_aspect_heads
from project.dataset import load_split
from project.preprocessing import save_enriched_split

test_raw = load_split(config, 'test')
test_enriched = save_enriched_split(test_raw, tokenizer, config, 'test')
aspect_results = evaluate_aspect_heads(hybrid_model, tokenizer, test_enriched, config)

import pandas as pd
display(pd.DataFrame(aspect_results['score_metrics']).T)
display(pd.DataFrame(aspect_results['span_metrics']).T)
display(pd.DataFrame(aspect_results['faithfulness_rates']).T)


## Ablation checklist

Suggested runs to compare against the full CE + score + span + faith objective (see `docs/ASPECT_ATTENTION_GUIDE.md` §7 for details):

1. `lambda_score=0, lambda_span=0, lambda_faith=0` (CE-only baseline)
2. `+lambda_score` only (Phase 1)
3. `+lambda_span` (Phase 1 + 2, no faithfulness)
4. Full objective (`+lambda_faith`)

Compare `outputs/reports/test_metrics.json` (generative) and `outputs/reports/test_metrics_aspect.json` (head-based) across runs.